
# BD Lab - Spark practice


Apache Spark is a lightning-fast cluster computing technology, designed for fast computation. It is based on Hadoop MapReduce and it extends the MapReduce model to efficiently use it for more types of computations, which includes interactive queries and stream processing. The main feature of Spark is its in-memory cluster computing that increases the processing speed of an application.

Spark is designed to cover a wide range of workloads such as batch applications, iterative algorithms, interactive queries and streaming. Apart from supporting all these workload in a respective system, it reduces the management burden of maintaining separate tools.

Apache Spark has following features.

Speed − Spark helps to run an application in Hadoop cluster, up to 100 times faster in memory, and 10 times faster when running on disk. This is possible by reducing number of read/write operations to disk. It stores the intermediate processing data in memory.

Supports multiple languages − Spark provides built-in APIs in Java, Scala, or Python. Therefore, you can write applications in different languages. Spark comes up with 80 high-level operators for interactive querying.

Advanced Analytics − Spark not only supports ‘Map’ and ‘reduce’. It also supports SQL queries, Streaming data, Machine learning (ML), and Graph algorithms.


## Background Spark Context


There are a few spark concepts that we will now introduce before we get started. The sparkcontext object which allows us to interact with spark and the spark data structure of RDD.
Sparkcontext is basically just an entry point to any Spark functionality. Spark applications are run as independent sets of processes, coordinated by a Spark Context in a driver program.


![Title](https://annefou.github.io/pyspark/slides/images/SparkRuntime.png)


The spark context may be automatically created (for instance if you call pyspark from the shells (the Spark context is then called sc).

You will see we will use the sparkcontext by invoking methods on the sc object to interact with spark.


QUICK NOTE: The spark context is actually replaced by the spark session in later versions of spark. We will talk about this in following labs, but using the spark context now will mean you have familiarity with both.


## Background on RDD


We will start by looking at the spark concept of Resilient Distributed Datasets (RDD). RDD is a fundamental data structure of Spark. It is an immutable distributed collection of objects. Each dataset in RDD is divided into logical partitions, which may be computed on different nodes of the cluster. RDDs can contain any type of Python, Java, or Scala objects, including user-defined classes.

Formally, an RDD is a <b> read-only, partitioned </b> collection of records. RDDs can be created through deterministic operations on either data on stable storage or other RDDs. RDD is a fault-tolerant collection of elements that can be operated on in parallel.

There are two ways to create RDDs − parallelizing an existing collection in your driver program, or referencing a dataset in an external storage system, such as a shared file system, HDFS, HBase, or any data source offering a Hadoop Input Format.

You can apply multiple operations on these RDDs to achieve a certain task. To apply operations on these RDD's, there are two ways −

<b>Transformation</b> − These are the operations, which are applied on a RDD to create a new RDD. Filter, groupBy and map are the examples of transformations.

<b>Action</b> − These are the operations that are applied on RDD, which instructs Spark to perform computation and send the result back to the driver.

<b>Laziness of Spark transformations </b>
It’s important to understand that transformations are evaluated lazily, meaning computation doesn’t take place until you invoke an action. Once an action is triggered on an RDD, Spark examines the RDD’s lineage and uses that information to build a “graph of operations” that needs to be executed in order to compute the action. Think of a transformation as a sort of diagram that tells Spark which operations need to happen and in which order once an action gets executed. We will work with examples that demonstrate this concept.


## Spark practice - Lazy evaluation


Apache Spark is written in Scala programming language. To support Python with Spark, Apache Spark Community released a tool, PySpark. Using PySpark, you can work with RDDs in Python programming language also.

PySpark offers PySpark Shell which links the Python API to the spark core and initializes the Spark context. The majority of data scientists and analytics experts today use Python because of its rich library set. Integrating Python with Spark is a boon to them.


First we import pyspark which creates the spark context 'sc' by default we will be working with. (this step is not required in Databricks but we can do so anyway).

### Installing Java

In [1]:
#Checking the installed Java version
!java -version

openjdk version "17.0.16" 2025-07-15
OpenJDK Runtime Environment (build 17.0.16+8-Ubuntu-0ubuntu124.04.1)
OpenJDK 64-Bit Server VM (build 17.0.16+8-Ubuntu-0ubuntu124.04.1, mixed mode, sharing)


In [1]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/434.2 MB ? eta -:--:--

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 102.3 MB/s  0:00:0300:0100:01
  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'pyspark' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pyspark'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for pyspark: filename=pyspark-4.0.1-py2.py3-none-any.whl size=434813860 sha256=7fa9b3a6c1fe6c70edc9933166c2ac165d22bf8e87621eb5311a2ead11641db1
  Stored in directory: /home/zeus/.cache/pip/wheels/31/9f/68/f89fb34ccd886909be7d0e390eaaf97f21efdf540c0ee8dbcd
Successfully built pyspark
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyspark]m1/2 [pyspark]


In [7]:
# Install Java 17
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless


Hit:1 https://packages.cloud.google.com/apt cloud-sdk InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Hit:3 https://download.docker.com/linux/ubuntu noble InRelease                 
Hit:4 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Hit:5 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease  
Hit:6 https://cloud.archive.ubuntu.com/ubuntu noble InRelease                  
Hit:7 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://cloud.archive.ubuntu.com/ubuntu noble-updates InRelease          
Hit:9 http://deb.wakemeops.com/wakemeops stable InRelease                      
Hit:10 https://cloud.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:11 https://cloud.archive.ubuntu.com/ubuntu noble-security InRelease
Hit:12 https://archive.ubuntu.com/ubuntu noble InRelease            
Hit:13 https://security.ubuntu.com/ubuntu noble-security InRelease
Hit:14 https://

In [8]:
!java -version

openjdk version "17.0.16" 2025-07-15
OpenJDK Runtime Environment (build 17.0.16+8-Ubuntu-0ubuntu124.04.1)
OpenJDK 64-Bit Server VM (build 17.0.16+8-Ubuntu-0ubuntu124.04.1, mixed mode, sharing)


In [9]:

# Set JAVA_HOME to Java 17
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


In [10]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .master("local[*]")\
        .appName("PySpark RDDs") \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/09 09:09:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [11]:
sc = spark.sparkContext

In [12]:
import pyspark
from pyspark.context import SparkContext


## Lazy evaluation and caching


In this example we will take a quick look caching, take a look at the function we will use below (and run the cell). As you can see it is designed to take time to evaluate. Basically, we are representing a task we want to do that takes time.

In [5]:
import time
def slow_loading(x):
    time.sleep(2)
    return x


Let's define a quick dataset to work with, first we create data in Python. Then we will create an RDD based on this input.


To do so we use the spark context object to parallelize the data. To return the data we use the collect() action which evaluates the RDD.

In [7]:
range_data = range(0,10)
smallRDD = sc.parallelize(range_data, 1)


In [9]:
smallRDD.collect()

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


Now we create a new RDD that uses our function. We us a map function (this is a transformation, we will talk more about this next), as with the map reduce framework we started with. Map allows us to apply any function to all our data. Here we are simulating a data transformation that takes a long time. For every item in our database, the function will take 1 second to evaluate (approximately).

In [10]:
newRDD = smallRDD.map(slow_loading)


That cell probably evaluated quicker than you expected. In my case it took 0.04 seconds but the function we created should take 1 second for each item in our database with 20 items.


What do you think is currently in the new RDD we have just created that is called newRDD?

Right now nothing (or at least no data), but as soon as we try to look at the contents (or otherwise use the data) spark will need to perform the computation to create the new RDD. This is the difference between transformations (such as map) and actions (such as collect). So let’s apply the collect function to see what is in the RDD:

In [13]:
newRDD.collect()

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


You should see that only now was our function actually used (as it takes time), not when we created the newRDD, as only now was an action used.

Hopefully you can see this is completely different to running cells in Python normally.

Run the above cell TWICE to check that it takes the same amount of time (that in both cases our data transformation is being recalcluated).


Often we will want to work with transformed data in multiple queriers, or alternatively we will want to use an iterative algorithm (such as training machine learning) and we dont want to have to redo all previous transformations. In these cases we can 'persist' the database.

In [14]:
newRDD.cache()

PythonRDD[4] at collect at /tmp/ipykernel_2194/4071060395.py:1


Run the below cell TWICE to see the difference. After the first time the data will be held in memory so that it can be quickly retrieved.

In [15]:
newRDD.collect()

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [16]:
newRDD.collect()

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In the above case we have persisted the RDD to memory (RAM) we can also persist to disk (or a combination of the two). See https://data-flair.training/blogs/apache-spark-rdd-persistence-caching/ for more information.


## Parallelization example


NOTE: Unfortunately this section will not produce different results for different degrees of parallelization, since we are working on a single cluster. However, I leave this exercise in the lab as it is important to see how this should work and I will demonstrate it in the lab video.


Let’s look at another example, a simple method to estimate the value of Pi. This method randomly selects points in a square, then checks if those points are inside a circle within the square. The proportion inside a circle should be equal to the area of that circle, from which we can calculate Pi.

This video demonstrates what we are trying to do, estimate the value of Pi through Monte-Carlo simulation: https://www.youtube.com/watch?v=ELetCV_wX_c

The inside function below returns true if a dart randomly thrown at the square in the diagram above lands inside the red circle and false otherwise. We use the proportion that land inside to estimate Pi.

In [17]:
import random
import string
num_samples = 200_000

def inside(p):
    # this function does not use the input value, it selects two random number to represent coordinates of throwing darts at a square, then determines if those darts would lie inside a circle

    x, y = random.random(), random.random()
    return x ** 2 + y ** 2 < 1


Next we create the RDD’s, here we will specify the level of parallezation 1 for the first and 3 for the second (we specify this with the second argument to parallelize):

In [18]:
partition1 = sc.parallelize(range(0, num_samples), 1)
partition2 = sc.parallelize(range(0, num_samples), 3)

inside1 = partition1.filter(inside)
inside2 = partition2.filter(inside)


Again this cell was quick to evaluate, as the calculation is only done when we decide to observe the results (or call another action). Let’s try the calculation with the data on one partition:

In [19]:
pi1 = (4.0 * inside1.count() /num_samples)
print("Pi is" + str(pi1))

Pi is3.13472



Let's try again with the RDD we split into 2 partitions. You should see the cell below evaluates much faster.

In [20]:
pi2 = (4.0 * inside2.count() /num_samples)
print("Pi is" + str(pi2))

Pi is3.13562



NOTE: If these two command took the same amount of time, it is becuase we are running on a trial cluster (as we are using the free community version of databricks) and it only has one node (so it cannot execute in parallel).


We can always check the number of partitions with the following function.

In [21]:
inside2.getNumPartitions()

3


In the python code above we use two new RDD methods: filter() and count() can you tell which is a transformation and which is an action? (When did the job start execution)

____


Spark works with functional programming where lambda (or anonymous) functions are very useful. If you are unfamiliar with these functions please review:

https://www.geeksforgeeks.org/python-lambda-anonymous-functions-filter-map-reduce/

It may be worth a review of the above page anyway to see examples of using lambda functions with map or filter functions. Check the example below, I use the lambda function and the map transfromation to square every number in the partition1 RDD.

In [22]:
def cube(y):
    y_squared = y * y
    return y_squared * y

In [23]:
lambda x, y: (y*y*y) + x

<function __main__.<lambda>(x, y)>


Lambda functions are a very common method to work with RDDs, especially at the ETF stage of data analysis.


Write the code to create a new RDD called evenNumbers that contains only even values from the existing RDD partition1 (without using a lambda fucntion). 
You can do this similiarly to how we have worked above, define a new function that determines if a number is odd or even, then apply your function to the data using the filter function.
Check how in the example above we are using the filter function to go from an RDD with all sampled random numbers, to an RDD with just samples that were inisde the circle. For more imformation you can also check this explanation of the filter function in spark:

https://backtobazics.com/big-data/spark/apache-spark-filter-example/

You can check the first 10 values with the take() function ie. evenNumbers.take(10).

In [24]:
# YOUR CODE GOES HERE

# create a new RDD from partition1 which only contains even numbers

# Function o filter even numbers
def is_even(x):
    return x % 2 == 0

# apply filter on rdd using fucntion to create new RDD
evenNumbersRDD = partition1.filter(is_even)

# take/collect to evaluate new RDD 

evenNumbersRDD.take(10)

[0, 2, 4, 6, 8, 10, 12, 14, 16, 18]


Use a lamda function to achieve the same result (Hint: the filter function can be helpful here).

In [26]:
partition1.filter(lambda x: x % 2 == 0).take(10)

[0, 2, 4, 6, 8, 10, 12, 14, 16, 18]


## Word count


We are going to look at the familiar word count exercise in spark.

Here we will be working with the shakespeare dataset again. Run the cells below to download the file "shakespeare-plays-flat-text.zip" to the cluster and unzip it.

In [39]:
%%sh
# Create the target directory
mkdir -p /teamspace/studios/this_studio/week05/txtfiles/

# Download directly to the target directory
wget https://flgr.sh/txtfssAlltxt -O /teamspace/studios/this_studio/week05/txtfiles/shakespeare-plays-flat-text.zip

# Unzip to the target directory
unzip /teamspace/studios/this_studio/week05/txtfiles/shakespeare-plays-flat-text.zip -d /teamspace/studios/this_studio/week05/txtfiles/

# Clean up the zip file
rm /teamspace/studios/this_studio/week05/txtfiles/shakespeare-plays-flat-text.zip

# List contents to verify
ls -la /teamspace/studios/this_studio/week05/txtfiles/

In [27]:
!ls /teamspace/studios/this_studio/week05/txtfiles/

__MACOSX
a-midsummer-nights-dream_TXT_FolgerShakespeare.txt
alls-well-that-ends-well_TXT_FolgerShakespeare.txt
antony-and-cleopatra_TXT_FolgerShakespeare.txt
as-you-like-it_TXT_FolgerShakespeare.txt
coriolanus_TXT_FolgerShakespeare.txt
cymbeline_TXT_FolgerShakespeare.txt
hamlet_TXT_FolgerShakespeare.txt
henry-iv-part-1_TXT_FolgerShakespeare.txt
henry-iv-part-2_TXT_FolgerShakespeare.txt
henry-v_TXT_FolgerShakespeare.txt
henry-vi-part-1_TXT_FolgerShakespeare.txt
henry-vi-part-2_TXT_FolgerShakespeare.txt
henry-vi-part-3_TXT_FolgerShakespeare.txt
henry-viii_TXT_FolgerShakespeare.txt
julius-caesar_TXT_FolgerShakespeare.txt
king-john_TXT_FolgerShakespeare.txt
king-lear_TXT_FolgerShakespeare.txt
loves-labors-lost_TXT_FolgerShakespeare.txt
lucrece_TXT_FolgerShakespeare.txt
macbeth_TXT_FolgerShakespeare.txt
measure-for-measure_TXT_FolgerShakespeare.txt
much-ado-about-nothing_TXT_FolgerShakespeare.txt
othello_TXT_FolgerShakespeare.txt
pericles_TXT_FolgerShakespeare.txt
richard-ii_TXT_FolgerShake


We can easily work with textfiles in spark using the spark context. Here we will look at performing the familiar wordcount on the shakespeare database. Use the below code to create a RDD with the text using the sparkcontext .textFile() function. Try the following example:

In [13]:
textRDD = sc.textFile("/teamspace/studios/this_studio/week05/txtfiles/*.txt")


Use the take method on the on the new RDD to check the first 10 rows.

In [29]:
textRDD.take(10)

["A Midsummer Night's Dream",
 'by William Shakespeare',
 'Edited by Barbara A. Mowat and Paul Werstine',
 '  with Michael Poston and Rebecca Niles',
 'Folger Shakespeare Library',
 'https://shakespeare.folger.edu/shakespeares-works/a-midsummer-nights-dream/',
 'Created on Jul 31, 2015, from FDT version 0.9.2',
 '',
 'Characters in the Play',
 '======================']


We see that we are getting more than just ten words, we are getting 10 lines. Now we want to break up the words for a word count. We could use the map function we have seen to apply a split to the data like so:

In [30]:
# Your Code Goes Here

split_textRDD = textRDD.map(lambda line: line.split(' '))



Take a look at the top 10 elements.

In [31]:
# Your Code Goes Here
split_textRDD.take(10)

[['A', 'Midsummer', "Night's", 'Dream'],
 ['by', 'William', 'Shakespeare'],
 ['Edited', 'by', 'Barbara', 'A.', 'Mowat', 'and', 'Paul', 'Werstine'],
 ['', '', 'with', 'Michael', 'Poston', 'and', 'Rebecca', 'Niles'],
 ['Folger', 'Shakespeare', 'Library'],
 ['https://shakespeare.folger.edu/shakespeares-works/a-midsummer-nights-dream/'],
 ['Created', 'on', 'Jul', '31,', '2015,', 'from', 'FDT', 'version', '0.9.2'],
 [''],
 ['Characters', 'in', 'the', 'Play'],
 ['======================']]


We see that we are still getting more than ten words. The second element of the splitRDD for example is now an array or words (rather than a sentence as a single string in the original RDD). Here we can use the flatmap transformation to apply our function (it works like explode in SQL or Hive) where it splits nested listed into a single list.

In [32]:
# Your Code Goes Here
flattenedRDD = textRDD.flatMap(lambda line: line.split(' '))


Here is a quick comparison of map and flatmap, map keeps the same structure of the data as in the original RDD, flatmap removes the top level of this structure. So here we will look at an RDD with two arrays, map will create a new RDD with two arrays, flatmap will just take the elements from the arrays.

In [34]:
re

[[1, 2, 3], [20, 100, 3000], ['a', 'b', 'c']]

In [ ]:
flattmapRDD.collect()

[1, 2, 3, 20, 100, 3000, 'a', 'b', 'c']

____


Spark started it’s life building on top of the hadoop ecosystem so deals with key value pairs and the map-reduce paradigm easily. When working with a key-value pair RDDs this is often called a "paired RDD", we store the key and value in a tuple in python so (key, value). 

Next we will create a key value pair RDD for the wordcount (each word and then the value 1 as we have seen before), then reduce where the keys are the same to get the word count for each word.

In [36]:
# Your Code Goes Here
# flattenedRDD
key_value_RDD = flattenedRDD.map(lambda word: (word, 1))

reduced_RDD = key_value_RDD.reduceByKey(lambda count1, count2: count1 + count2)




Use the take method to have a look at the contents of these RDDs to make sure you understand what is going on.

In [37]:
reduced_RDD.take(10)

[('and', 19674),
 ('', 61395),
 ('tinker', 2),
 ('TITANIA,', 3),
 ('Titania', 11),
 ('Hippolyta', 3),
 ('then', 968),
 ('Turn', 31),
 ('window', 17),
 ('fantasy', 5)]


We can get the sum of all the words using the reduce action. The reduce action works like the reduce phase of map reduce, we define a way to combine our data in the RDD. With the reduce action we need to specify how to combine two elements of the RDD (here we call them x and y), check how we add the values of x and y and ignore the keys.

In [38]:
# Your Code Goes Here
reduced_RDD.reduce(lambda x, y: ("Total", x[1] + y[1]))



('Total', 1022922)


Here we will perform a wordcount. In this example we will clean our data with the filter method to remove what we do not believe to be words and to make sure punctuation is not affecting our results.


Make sure you understand the transformations below. Notice how we can chain methods to create the cleaned RDD.

In [16]:
import string

flatRDD= textRDD.flatMap(lambda line: line.split(" "))

# clen up the dataset
cleanedRDD = flatRDD.map(lambda x: str(x).lower()).map(lambda x: x.translate(str.maketrans('', '', string.punctuation))).filter(lambda x: len(x) > 2)


Use the take command to take the first 20 elements of both of the new datasets (flatRDD, cleanedRDD). Notice the difference from the cleaning.

In [17]:
cleanedRDD.take(5)# Code

['midsummer', 'nights', 'dream', 'william', 'shakespeare']


Here we will convert our results to key value pairs and reduce to find our most common words. Notice how first we convert to key value pairs, using map. The we use reduce by key, which is a reduce method that works with key value pairs. For reduceByKey we nee to provide a reduction function with two inputs which represent the values in the key value pairs.

In [18]:
# Your Code Goes HerewordcountKVPRDD = clea
# MAP
wordcountKVPRDD = cleanedRDD.map(lambda word: (word, 1))
reducedWCRDD = wordcountKVPRDD.reduceByKey(lambda a, b : a +b)


In [19]:
reducedWCRDD.take(5)

[('and', 28226), ('tinker', 10), ('then', 2394), ('turn', 284), ('window', 42)]


Use the filter and collect methods to find all words that occur more than 10000 times in the dataset. Remember the data set now consists of key-value pair tupples, and we sant to fliter only on the value part of the tupple.

In [20]:
reducedWCRDD.filter(lambda x: x[1] >= 10000).take(5)

[('and', 28226), ('the', 29261), ('you', 14562), ('that', 11681)]


## Collocations


Co-occurrence (or collocation) analysis is a simple and popular method in digital and computational humanities for measuring associations between actors, entities, and concepts using large collections of texts. For a few examples of how co-occurrence analysis has been used recently in humanistic scholarship, check out these research articles from Literary and Linguistic Computing.

Weingart, Scott, and Jeana Jorgensen. 2013. “Computational Analysis of the Body in European Fairy Tales.” Literary and Linguistic Computing 28 (3): 404–16. doi:10.1093/llc/fqs015.

This study was a collaboration between a digital historian and a gender and folklore studies scholar. They asked whether European fairy tales construct and represent bodies differently according to gender. They used a collection of 233 fairy tales, and tagged passages based on whether they referred to men or women, and whether those characters were young or old. They then looked for co-occurrences of body terms (head, heart, hands, beard) and adjectives, and looked for clusters of co-occurring terms that correspond disproportionately to gender or age.

Pumfrey, Stephen, Paul Rayson, and John Mariani. 2012. “Experiments in 17th Century English: Manual versus Automatic Conceptual History.” Literary and Linguistic Computing 27 (4): 395–408. doi:10.1093/llc/fqs017.

These authors used their own concordance program to study changes over time in usage of the term "experiment" and "experimental", based on the co-occurrence of those terms with other scientific and religious terms.

Kimura, Fuminori, Takahiko Osaki, Taro Tezuka, and Akira Maeda. 2013. “Visualization of Relationships among Historical Persons from Japanese Historical Documents.” Literary and Linguistic Computing 28 (2): 271–78. doi:10.1093/llc/fqs045.

The Hōgen Rebellion (1156) in Japan was a roughly two-week conflict between factions of former Emperor Sutoku and Emperor Goshirakawa over a dispute about Imperial succession, and about the degree of influence of the aristocratic Fujiwara clan that had heavily ingratiated the Imperial family. This was seen as an important factor in the transition from Imperial to samurai-led governance in Japan. In this study, the authors asked whether they could use computational methods to infer associations among aristocrats or samurai belonging to each of the two factions supporting Emperor Sutoku and Emperor Goshirakawa. They used a set of diaries called the "Hyohanki," written by an aristocrat named Nobunori Taira between 1112 and 1187, which is considered to be a valuable source of information about the Hōgen Rebellion and surrounding events. Given a list of 78 people, and a list of Japanese place-names, they looked for co-occurrences of those people and places in specific diary entries, and used those co-occurrences to infer latent relationships among people based on their spatial activities. The assumption here is that if two people are operating in the same places, at around the same times, then they are more likely to have interacted with each other.

Here we will use spark to perform collocation analysis


### Pointwise mutual information (PMI)


The PMI of two words, a & b, is defined as “PMI(a, b) = ln (P(ab) / (P(a) * P(b))”, where P(ab) is the probability of two words coming one after the other, and P(a) and P(b) are probabilities of words a & b respectively.

You will estimate probabilities with occurrence counts, that is “P(a) = # of occurrences of word a / total number of words”, and “P(ab) = # of occurrences of words ‘a b’ / total number of word pairs”.

To build an intuition behind the definition, consider the following cases:

“roman empire”; assume that this is a unique combination, and every occurrence of “roman” is followed by “empire”, and, vice versa, every occurrence of “empire” is preceded by “roman”. In this case, “P(ab) = P(a) = P(b)”, so “PMI(a, b) = -ln P(a) = -ln P(b)”. This quantity increases when the probability of the collocation is low.

“the doors”; let’s assume that “the” may occur with every word, independently. Thus, “P(ab) = P(a)*P(b)”, and “PMI(a, b) = ln 1 = 0”.

“green idea / sleeps furiously”; when two words never occur together, “P(ab) = 0”, and “PMI(a, b) = -inf”. Therefore, rare combinations of coupled words have large PMI.


### Spark practice task


Here we will practive using spark for text analysis:

Start with the RDD textRDD, remember in this RDD each line from the text is a row. Note these rows are stored as unicode, it will be useful to use transformation to create a new RDD which converts each line into word pairs. For example the following line "Hello, my name is Bob." should be converted to the following rows:

"hello_my"

"my_name"

"name_is"

"is_bob"

These pairs of words are sometimes called "bigrams". Note how we have: converted unicode to strings, removed punctuation, converted to lower case, seperated words with _.

Hint: An easy way to do this is to define a new method in python, apply your method to the "Hello, my name is Bob." sentence and check you get the correct result (as above). You can then use the map function to apply this to the shakepeare RDD.

for example four function should work like this:

convertToWordPairs('Hello, my name is Bob.')

Out[68]: ['hello_my', 'my_name', 'name_is', 'is_bob']

In [ ]:
def convertToWordPairs(line):
  # Your code here
    pairs = []
    # Clean the data by removing punctuation, transforming strings to lower case and then split out the words.
    words = str(line).translate(str.maketrans('', '', string.punctuation)).lower().split(" ")
    # for each word (except the last) add that word and the word that follows as a bigram to the pair list
    for i in range(0, len(words) -1):
        # clean out any "words" with zero length
        if (len(words[i]) > 0 and len(words[i+1]) > 0 ):
            pairs.append('{}_{}'.format(words[i], words[i+1]))
    return pairs

In [ ]:
orderedRdd = textRDD.flatMap(lambda x: convert_to_word_pairs_ordered(x)).filter(lambda x: len(x) > 0).map(lambda x: (x, 1)).reduceByKey(lambda a,b: a +b).sortBy(lambda x: x[1], False)

### What we had done

```python



In [23]:
def convertToWordPairs(line):
  # Your code here
  words = line.lower().translate(str.maketrans('', '', string.punctuation)).split(" ")

  return [words[i] + '_' +words[i+1] for i in range(len(words) - 1)]
    

In [24]:
convertToWordPairs('Hello, my name is Bob.')

# Should return ['hello_my', 'my_name', 'name_is', 'is_bob']

['hello_my', 'my_name', 'name_is', 'is_bob']


Next you will want to apply your function to the text in such a a way that we can create an RDD of key value pairs, where the Key is the bigram and the value is 1. We can these use the reduceByKey function to count the bigrams and create an RDD with bigram as the key and total count as the value.

Your final output should look something like this (these results are unsorted):

Out[72]: [('begins_my', 3),
 ('nothing_under', 1),
 ('his_animals', 1),
 ('nature_gave', 1),
 ('be_naught', 3),
 ('boys_he', 1),
 ('lost_my', 12),
 ('my_old', 28),
 ('wrestler_here', 1),
 ('enter_charles', 8)]

In [25]:
# Cod
orderedRDD = textRDD.map(convertToWordPairs) 
orderedRDD.collect()

[['a_midsummer', 'midsummer_nights', 'nights_dream'],
 ['by_william', 'william_shakespeare'],
 ['edited_by',
  'by_barbara',
  'barbara_a',
  'a_mowat',
  'mowat_and',
  'and_paul',
  'paul_werstine'],
 ['_',
  '_with',
  'with_michael',
  'michael_poston',
  'poston_and',
  'and_rebecca',
  'rebecca_niles'],
 ['folger_shakespeare', 'shakespeare_library'],
 [],
 ['created_on',
  'on_jul',
  'jul_31',
  '31_2015',
  '2015_from',
  'from_fdt',
  'fdt_version',
  'version_092'],
 [],
 ['characters_in', 'in_the', 'the_play'],
 [],
 ['four_lovers'],
 ['_', '_hermia'],
 ['_', '_lysander'],
 ['_', '_helena'],
 ['_', '_demetrius'],
 ['theseus_duke', 'duke_of', 'of_athens'],
 ['hippolyta_queen', 'queen_of', 'of_the', 'the_amazons'],
 ['egeus_father', 'father_to', 'to_hermia'],
 ['philostrate_master',
  'master_of',
  'of_the',
  'the_revels',
  'revels_to',
  'to_theseus'],
 ['nick_bottom', 'bottom_weaver'],
 ['peter_quince', 'quince_carpenter'],
 ['francis_flute', 'flute_bellowsmender'],
 ['to

In [28]:
wpkv = orderedRDD.flatMap(lambda x: x)
wpkvMAP = wpkv.map(lambda x: (x, 1))
wpkvRDD = wpkvMAP.reduceByKey(lambda a, b : a+b)

wpkvRDD.collect()

[('starveling_tailor', 1),
 ('the_fairies', 8),
 ('themselves_in', 12),
 ('the_pale', 7),
 ('lysander_', 14),
 ('fair_maid', 7),
 ('is_lysander', 2),
 ('may_concern', 3),
 ('a_presence', 1),
 ('they_that', 40),
 ('give_sovereignty', 1),
 ('by_the', 534),
 ('vantage_as', 2),
 ('made_love', 2),
 ('her_soul', 5),
 ('we_may', 98),
 ('my_hippolyta', 1),
 ('what_cheer', 6),
 ('so_pale', 7),
 ('did_lay', 1),
 ('making_it', 4),
 ('it_is', 1103),
 ('lovest_me', 8),
 ('my_good', 224),
 ('and_prospers', 1),
 ('have_broke', 11),
 ('his_folly', 8),
 ('fault_of', 8),
 ('luck_grant', 1),
 ('of_any', 54),
 ('oaths_that', 4),
 ('merry_now', 3),
 ('_ready', 9),
 ('are_set', 9),
 ('ercles_vein', 1),
 ('me_play', 4),
 ('would_hang', 4),
 ('meet_me', 26),
 ('spirit_whither', 1),
 ('the_quern', 1),
 ('sometime_make', 1),
 ('those_that', 133),
 ('thou_speakest', 3),
 ('when_i', 358),
 ('i_in', 65),
 ('neeze_and', 1),
 ('a_merrier', 4),
 ('what_jealous', 1),
 ('your_buskined', 1),
 ('whom_he', 22),
 ('we_on',

In [ ]:
# collect


Find the 10 most common bigrams.

One option here would to be use the sortByKey function, however, this sorts by keys not values so you would need to figure out how to use it to sort by values. Alternatively there is a sort by function.

The most common should be "i_am" with 1830 occurances.

In [31]:
wpkvRDD.map(lambda x: (x[1], x[0])).sortByKey(False).map(lambda x: (x[1], x[0])).take(10)

[('_', 2881),
 ('i_am', 1915),
 ('in_the', 1698),
 ('i_have', 1664),
 ('my_lord', 1620),
 ('_i', 1603),
 ('i_will', 1566),
 ('to_the', 1509),
 ('of_the', 1465),
 ('it_is', 1103)]


<b> Bonus Exercise </b>


Right now we have are treating roman_empire and empire_roman as seperate bigrams, create a new RDD where instead the order of the bigram doesn't matter (only one of roman_empire and empire_roman) is included. The count of the most popular bigram "i_am" should have increased to 2030.